# CartFlow (P06) — Week 8: Gold Hand-off to Power BI

**Official notebook:** `notebooks/06_powerbi_export.ipynb`

**Purpose:** Validate the approved CartFlow Gold batch summaries, define the controlled Power BI hand-off, prove the hand-off preserves the Gold grain and selected business measures, and document the Power BI connection boundary.

**Gold-only rule:** Power BI must use approved Gold outputs only. Do not connect to raw, Bronze, Silver Candidate, Trusted Silver detail, or Quarantine data.

**Selected Gold batch summaries**
- `agg_sales_daily`
- `agg_seller_performance`
- `agg_category_sales`
- `agg_delivery_delay`
- `agg_payment_review`

Each Gold summary remains an independent source. Do not flatten the five Gold grains into one master table.


## Week 8 workflow

1. Gold source register
2. Gold hand-off validation
3. Controlled Power BI hand-off definitions
4. Gold-to-hand-off reconciliation
5. Repeat-run business-content fingerprint
6. Power BI connection manifest
7. Final hand-off checkpoint

**Important:** PASS/FAIL conclusions must come from actual Databricks execution. Do not manually enter results.


In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


## 1. Gold source register

| Gold table | Declared grain | Key | Main measures / purpose | Power BI use |
|---|---|---|---|---|
| `agg_sales_daily` | One row per sales date | `sales_date` | `total_orders`, `non_cancelled_orders`, `gmv`, `freight`, `aov` | Sales overview and daily trend |
| `agg_seller_performance` | One row per seller | `seller_id` | `order_count`, `cancelled_orders`, `returned_orders`, `gmv`, `average_review_score` | Seller performance |
| `agg_category_sales` | One row per month + category | `year_month + category_code` | `item_lines`, `order_count`, `gmv`, `item_total`, `freight`, `average_order_value` | Category sales |
| `agg_delivery_delay` | One row per purchase date + customer state | `purchase_date + customer_state` | `delivery_eligible_orders`, `on_time_orders`, `average_delay_days`, `on_time_delivery_rate` | Delivery performance |
| `agg_payment_review` | One row per currency | `currency_code` | `reconciled_orders`, `complete_coverage_orders`, `payment_reconciliation_rate`, `average_review_score`, `reviewed_order_coverage` | Payment/review quality |


In [ ]:
%sql
SHOW TABLES LIKE 'agg_*';


In [ ]:
%sql
SELECT 'agg_sales_daily' AS gold_table, COUNT(*) AS rows FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance', COUNT(*) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales', COUNT(*) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay', COUNT(*) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review', COUNT(*) FROM agg_payment_review
ORDER BY gold_table;


### Source-register checkpoint

Confirm from the results:
- `workspace.default` is the intended Gold location.
- All five approved batch-summary tables exist.
- Table names match the Week-7 Gold model.
- Row counts are actual Databricks results.

Do not manually type or invent row counts.


## 2. Gold hand-off validation

Verify that:
- report-facing columns exist;
- declared grain keys are unique;
- declared keys are not null/blank;
- selected business measures are populated where expected.


In [ ]:
%sql
WITH expected AS (
  SELECT 'agg_sales_daily' AS gold_table, column_name FROM VALUES
    ('sales_date'),('total_orders'),('non_cancelled_orders'),('gmv'),('freight'),('aov') AS v(column_name)
  UNION ALL SELECT 'agg_seller_performance', column_name FROM VALUES
    ('seller_id'),('order_count'),('cancelled_orders'),('returned_orders'),('gmv'),('average_review_score') AS v(column_name)
  UNION ALL SELECT 'agg_category_sales', column_name FROM VALUES
    ('year_month'),('category_code'),('item_lines'),('order_count'),('gmv'),('item_total'),('freight'),('average_order_value') AS v(column_name)
  UNION ALL SELECT 'agg_delivery_delay', column_name FROM VALUES
    ('purchase_date'),('customer_state'),('delivery_eligible_orders'),('on_time_orders'),('average_delay_days'),('on_time_delivery_rate') AS v(column_name)
  UNION ALL SELECT 'agg_payment_review', column_name FROM VALUES
    ('currency_code'),('reconciled_orders'),('complete_coverage_orders'),('payment_reconciliation_rate'),('average_review_score'),('reviewed_order_coverage') AS v(column_name)
),
actual AS (
  SELECT table_name AS gold_table, column_name
  FROM information_schema.columns
  WHERE table_schema=current_schema()
    AND table_name IN ('agg_sales_daily','agg_seller_performance','agg_category_sales','agg_delivery_delay','agg_payment_review')
)
SELECT e.gold_table,e.column_name AS expected_column,
       CASE WHEN a.column_name IS NOT NULL THEN 'PRESENT' ELSE 'MISSING' END AS column_status
FROM expected e LEFT JOIN actual a
ON e.gold_table=a.gold_table AND e.column_name=a.column_name
ORDER BY e.gold_table,e.column_name;


In [ ]:
%sql
SELECT 'agg_sales_daily' AS gold_table,COUNT(*) AS rows,COUNT(DISTINCT sales_date) AS distinct_keys,
COUNT(*)-COUNT(DISTINCT sales_date) AS duplicate_rows FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),COUNT(DISTINCT seller_id),COUNT(*)-COUNT(DISTINCT seller_id) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),COUNT(DISTINCT CONCAT(COALESCE(CAST(year_month AS STRING),''),'||',COALESCE(CAST(category_code AS STRING),''))),
COUNT(*)-COUNT(DISTINCT CONCAT(COALESCE(CAST(year_month AS STRING),''),'||',COALESCE(CAST(category_code AS STRING),''))) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),COUNT(DISTINCT CONCAT(COALESCE(CAST(purchase_date AS STRING),''),'||',COALESCE(CAST(customer_state AS STRING),''))),
COUNT(*)-COUNT(DISTINCT CONCAT(COALESCE(CAST(purchase_date AS STRING),''),'||',COALESCE(CAST(customer_state AS STRING),''))) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),COUNT(DISTINCT currency_code),COUNT(*)-COUNT(DISTINCT currency_code) FROM agg_payment_review
ORDER BY gold_table;


In [ ]:
%sql
SELECT 'agg_sales_daily' AS gold_table,SUM(CASE WHEN sales_date IS NULL THEN 1 ELSE 0 END) AS null_key_rows FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',SUM(CASE WHEN seller_id IS NULL OR TRIM(CAST(seller_id AS STRING))='' THEN 1 ELSE 0 END) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',SUM(CASE WHEN year_month IS NULL OR category_code IS NULL OR TRIM(CAST(category_code AS STRING))='' THEN 1 ELSE 0 END) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',SUM(CASE WHEN purchase_date IS NULL OR customer_state IS NULL OR TRIM(CAST(customer_state AS STRING))='' THEN 1 ELSE 0 END) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',SUM(CASE WHEN currency_code IS NULL OR TRIM(CAST(currency_code AS STRING))='' THEN 1 ELSE 0 END) FROM agg_payment_review
ORDER BY gold_table;


### Validation checkpoint

Expected structural results:
- required columns = `PRESENT`;
- duplicate grain rows = `0`;
- null/blank key rows = `0`.

If a check fails, investigate the upstream Gold output. Do not manually edit the hand-off.


## 3. Controlled Power BI hand-off definitions

Keep each source separate and preserve its declared Gold grain.


In [ ]:
%sql
SELECT sales_date,total_orders,non_cancelled_orders,gmv,freight,aov
FROM agg_sales_daily ORDER BY sales_date;


In [ ]:
%sql
SELECT seller_id,order_count,cancelled_orders,returned_orders,gmv,average_review_score
FROM agg_seller_performance ORDER BY seller_id;


In [ ]:
%sql
SELECT year_month,category_code,item_lines,order_count,gmv,item_total,freight,average_order_value
FROM agg_category_sales ORDER BY year_month,category_code;


In [ ]:
%sql
SELECT purchase_date,customer_state,delivery_eligible_orders,on_time_orders,average_delay_days,on_time_delivery_rate
FROM agg_delivery_delay ORDER BY purchase_date,customer_state;


In [ ]:
%sql
SELECT currency_code,reconciled_orders,complete_coverage_orders,payment_reconciliation_rate,average_review_score,reviewed_order_coverage
FROM agg_payment_review ORDER BY currency_code;


## 4. Gold-to-hand-off reconciliation

For the direct Databricks Power BI connection, the governed hand-off is the approved Gold table itself. These checks establish the baseline values Power BI must reproduce.


In [ ]:
%sql
SELECT 'agg_sales_daily' AS gold_source,COUNT(*) AS gold_rows,COUNT(*) AS handoff_rows,0 AS row_count_difference FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),COUNT(*),0 FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),COUNT(*),0 FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),COUNT(*),0 FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),COUNT(*),0 FROM agg_payment_review
ORDER BY gold_source;


In [ ]:
%sql
SELECT 'agg_sales_daily' AS gold_source,COUNT(*) AS rows,SUM(total_orders) AS total_orders,SUM(non_cancelled_orders) AS non_cancelled_orders,SUM(gmv) AS total_gmv,SUM(freight) AS total_freight FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),SUM(order_count),SUM(cancelled_orders),SUM(returned_orders),SUM(gmv) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),SUM(item_lines),SUM(order_count),SUM(gmv),SUM(item_total) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),SUM(delivery_eligible_orders),SUM(on_time_orders),SUM(average_delay_days),SUM(on_time_delivery_rate) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),SUM(reconciled_orders),SUM(complete_coverage_orders),SUM(payment_reconciliation_rate),SUM(reviewed_order_coverage) FROM agg_payment_review
ORDER BY gold_source;


## 5. Repeat-run business-content fingerprint

Run this cell once. Then rerun the unchanged Gold snapshot. The row counts and business fingerprints should remain unchanged.

Do not manually enter a matching value.


In [ ]:
%sql
SELECT 'agg_sales_daily' AS gold_source,COUNT(*) AS rows,
SUM(xxhash64(CAST(sales_date AS STRING),CAST(total_orders AS STRING),CAST(non_cancelled_orders AS STRING),CAST(gmv AS STRING),CAST(freight AS STRING),CAST(aov AS STRING))) AS business_fingerprint
FROM agg_sales_daily
UNION ALL SELECT 'agg_seller_performance',COUNT(*),
SUM(xxhash64(CAST(seller_id AS STRING),CAST(order_count AS STRING),CAST(cancelled_orders AS STRING),CAST(returned_orders AS STRING),CAST(gmv AS STRING),CAST(average_review_score AS STRING))) FROM agg_seller_performance
UNION ALL SELECT 'agg_category_sales',COUNT(*),
SUM(xxhash64(CAST(year_month AS STRING),CAST(category_code AS STRING),CAST(item_lines AS STRING),CAST(order_count AS STRING),CAST(gmv AS STRING),CAST(item_total AS STRING),CAST(freight AS STRING),CAST(average_order_value AS STRING))) FROM agg_category_sales
UNION ALL SELECT 'agg_delivery_delay',COUNT(*),
SUM(xxhash64(CAST(purchase_date AS STRING),CAST(customer_state AS STRING),CAST(delivery_eligible_orders AS STRING),CAST(on_time_orders AS STRING),CAST(average_delay_days AS STRING),CAST(on_time_delivery_rate AS STRING))) FROM agg_delivery_delay
UNION ALL SELECT 'agg_payment_review',COUNT(*),
SUM(xxhash64(CAST(currency_code AS STRING),CAST(reconciled_orders AS STRING),CAST(complete_coverage_orders AS STRING),CAST(payment_reconciliation_rate AS STRING),CAST(average_review_score AS STRING),CAST(reviewed_order_coverage AS STRING))) FROM agg_payment_review
ORDER BY gold_source;


## 6. Power BI connection manifest

Connect Power BI only to these five approved Gold tables in `workspace.default`:

- `agg_sales_daily`
- `agg_seller_performance`
- `agg_category_sales`
- `agg_delivery_delay`
- `agg_payment_review`

Keep them separate.

Do not create a combined master Gold table or direct summary-to-summary relationships merely because field names are similar.


## 7. Final hand-off checkpoint

Before treating the Gold hand-off as complete, confirm actual Databricks results show:
- all five Gold tables exist;
- required columns are present;
- declared grain keys are unique;
- keys have no null/blank values;
- selected measures are recorded;
- row counts reconcile;
- repeat-run fingerprints remain unchanged.

### Evidence to capture
Capture readable screenshots of:
1. Gold source inventory and row counts;
2. hand-off validation;
3. Gold-to-hand-off reconciliation;
4. repeat-run fingerprint.

Then preserve the Power BI model and dashboard evidence separately.

**Do not mark a check PASS unless the Databricks result actually supports it.**
